# 03. Legacy 전이 압축 DB materialization (Step 1 실행 금지)

이 노트북은 legacy 02의 LFW-development 압축기를 DB에 materialize하는 과거 시스템 경로입니다. Step 1은 SurvFace training development에서 독립 PCA/PQ를 fit하고 06의 fallback-free 평가 API로 공식 test를 분석합니다.

| 모드 | 예상 시간 |
| --- | ---: |
| `EXECUTE_STAGE=False` | 1초 미만 |
| 전체 변환·DB 저장·인덱스 | 약 30분~2시간 이상 |

> **진행/체크포인트/재시작**: batch별 DB commit과 30초 heartbeat를 사용합니다. 중단되면 Kernel Restart 후 처음부터 실행하면 동일 model hash/provenance 행은 건너뜁니다. 다른 compressor hash가 이미 섞여 있으면 자동 교체하지 않고 중단해야 하며, 그 경우 00부터 새 run을 만듭니다.


In [ ]:
# Step 1 실행 범위: 이 셀의 세 값만 바꾸고 Kernel Restart -> Run All
MODE = 'dev'             # 'dev' 또는 'real'
DATA_FRACTION = 1.0     # 0 < DATA_FRACTION <= 1
SEED = 42

import sys
from pathlib import Path

for _scope_root in (Path.cwd(), *Path.cwd().parents):
    if (_scope_root / 'research').is_dir():
        break
else:
    raise FileNotFoundError('D:/ronbun 내부에서 노트북을 실행하십시오.')
if str(_scope_root) not in sys.path:
    sys.path.insert(0, str(_scope_root))

from research.compression import PCA_SWEEP_DIMENSIONS
from research.experiments.scope import ExperimentScope

PCA_DIMENSIONS = (384, 256, 128, 64, 32)
PQ_SOURCE_DIMENSION = 512
if PCA_DIMENSIONS != tuple(PCA_SWEEP_DIMENSIONS):
    raise RuntimeError('노트북 PCA sweep과 공통 압축 정의가 다릅니다.')
EXPERIMENT_SCOPE = ExperimentScope(
    mode=MODE, data_fraction=DATA_FRACTION, seed=SEED
)
EXPERIMENT_SCOPE.as_dict()


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("D:/ronbun 내부에서 노트북을 실행하십시오.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from research.runtime import ProgressReporter, RunStore, resolve_active_run

EXECUTE_STAGE = True
EXECUTE_LEGACY_TRANSFER_MATERIALIZATION = False
if EXECUTE_STAGE or EXECUTE_LEGACY_TRANSFER_MATERIALIZATION:
    raise RuntimeError(
        "Legacy LFW compressor 전이 materialization은 Step 1 주 실험이 아닙니다. "
        "06_step1_compression_characterization.ipynb를 실행하십시오."
    )
BATCH_SIZE = 512
RUN_ROOT = PROJECT_ROOT / "runs" / "survface"

def resolve_run_for_preflight():
    try:
        return resolve_active_run(
            RUN_ROOT, environment_variable="RONBUN_SURVFACE_RUN_DIR"
        ), None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"

RUN_DIR, RUN_RESOLUTION_ERROR = resolve_run_for_preflight()
PROGRESS = ProgressReporter("SurvFace 03 compressed materialization", heartbeat_seconds=30)


## 1. transfer artifact preflight

현재 공통 helper가 target test에서 development 통계를 다시 계산하려 한다면 사용하면 안 됩니다. 이 노트북은 `frozen_error_stats`를 받는 전용 API만 호출합니다.


In [ ]:
preflight = {
    "execute_stage": EXECUTE_STAGE,
    "run_dir": str(RUN_DIR) if RUN_DIR else None,
    "run_resolution_error": RUN_RESOLUTION_ERROR,
    "batch_size": BATCH_SIZE,
    "required_api": "research.experiments.materialize_compressed_embeddings_with_frozen_stats",
}
preflight


## 2. 외부 통계 고정 변환

필요 API가 아직 구현되지 않았다면 명시적으로 중단합니다. 임시로 SurvFace test 통계를 fit하거나 가짜 artifact를 만들지 않습니다.


In [ ]:
def latest_transfer_manifest(run: RunStore) -> tuple[Path, dict]:
    run.verify_phase_artifacts("02_external_compressor_import")
    artifact_dir = run.run_dir / "artifacts" / "02_external_compressor_import"
    candidates = sorted(artifact_dir.glob("external_compressor_manifest_A*.json"))
    if not candidates:
        raise RuntimeError("02 external compressor manifest가 없습니다.")
    path = candidates[-1]
    return path, json.loads(path.read_text(encoding="utf-8"))


result = {"status": "not_executed", **preflight}
if EXECUTE_STAGE:
    if RUN_DIR is None:
        raise RuntimeError(f"SurvFace run을 찾지 못했습니다: {RUN_RESOLUTION_ERROR}")

    import research.experiments as experiment_api
    from research.compression import PCACompressor, PQCompressor
    from research.database import create_database_engine, init_database, load_database_settings
    from research.runtime.hashing import sha256_file

    materialize = getattr(
        experiment_api, "materialize_compressed_embeddings_with_frozen_stats", None
    )
    if materialize is None:
        raise NotImplementedError(
            "frozen_error_stats를 받는 공통 materialization API가 아직 없습니다. "
            "기존 development_image_paths API를 SurvFace official test에 사용하지 마십시오."
        )

    run = RunStore.open(RUN_DIR)
    run.verify_inputs()
    transfer_path, transfer = latest_transfer_manifest(run)
    if transfer.get("fit_split") != "development" or transfer.get("fit_on_survface_official_test"):
        raise ValueError("외부 development fit provenance가 아닙니다.")
    pca_path = run.run_dir / transfer["pca_artifact"]
    pq_path = run.run_dir / transfer["pq_artifact"]
    if sha256_file(pca_path) != transfer["pca_sha256"] or sha256_file(pq_path) != transfer["pq_sha256"]:
        raise ValueError("가져온 compressor artifact hash가 변경되었습니다.")
    pca = PCACompressor.load(pca_path)
    pq = PQCompressor.load(pq_path)
    engine = create_database_engine(load_database_settings())
    init_database(engine)

    with run.phase("03_official_compressed_materialization_and_index") as phase:
        suffix = f"A{phase.attempt:03d}"
        measurements_path = phase.attempt_dir / f"compression_measurements_{suffix}.csv"

        def report(message: str, details: dict[str, object]) -> None:
            PROGRESS.emit(message, **details)

        summary = materialize(
            engine,
            run_uid=run.run_id,
            pca=pca,
            pq=pq,
            pca_artifact_path=pca_path,
            pca_artifact_sha256=transfer["pca_sha256"],
            pq_artifact_path=pq_path,
            pq_artifact_sha256=transfer["pq_sha256"],
            error_normalization=transfer["error_normalization"],
            measurements_path=measurements_path,
            batch_size=BATCH_SIZE,
            progress=report,
        )
        summary.update({
            "source_fit_split": "development",
            "target_transform_split": "test",
            "fit_on_survface_official_test": False,
            "transfer_manifest": str(transfer_path),
        })
        summary_path = phase.attempt_dir / f"materialization_summary_{suffix}.json"
        summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
        phase.publish_artifact(measurements_path)
        phase.publish_artifact(summary_path)
        phase.record_counts(**summary["counts"])
    PROGRESS.emit("03 완료", **summary["counts"])
    result = {"status": "completed", "run_id": run.run_id, **summary}
else:
    PROGRESS.emit("검토 모드 완료: 변환/DB 저장/index 생성을 실행하지 않음", expected="1초 미만")
result


## 다음 단계

PCA retrieval vector는 256D pgvector 공간, certification용 reconstruction은 별도 512D 공간입니다. PQ code는 보조 artifact이며 pgvector 검색 벡터로 주장하지 않습니다. source/target split과 model hash가 맞는지 확인한 뒤 04로 이동합니다.
